In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

import pandas_datareader.data as web

In [ ]:
# Подготовка датасета для обучения в формате inputs, outputs
def make_datasets(input_data, n_inputs=2, n_outputs=1, gap=0):
	L = len(input_data)
	y = np.full((L-n_inputs-n_outputs-gap, n_outputs), 0.0)
	X = np.full((L-n_inputs-n_outputs-gap, n_inputs), 0.0)

	for i in range(n_inputs):
		X[:,i] = input_data[i:L-n_inputs-n_outputs-gap+i]

	for i in range(n_outputs):
		y[:,i] = input_data[n_inputs+gap+i:L-n_outputs+i]

	return X, y


In [ ]:
rate = web.DataReader(name='WGS10YR', data_source='fred', start='2000-01-01').diff().dropna()
rate.plot()
plt.show()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# Переведем ряда массив Numpy
series = rate.squeeze().to_numpy()

In [ ]:
# задаём ширину окна и горизонт прогнозирования
n_lags, fh= 20, 10

X, y = make_datasets(series, n_inputs=n_lags, n_outputs=fh)

In [ ]:
X_tensor = torch.Tensor(X).to(device)
y_tensor = torch.Tensor(y).to(device)

train_dataset = TensorDataset(X_tensor, y_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
def train(dataloader, model, loss_fn, optimizer, print_loss=True):
    size = len(dataloader.dataset)

    model.train()                                 # переводим молель в режим обучения
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()                     # Очищаем старые градиенты
        # Compute prediction error
        y_pred = model(X)                         # строми прооноз на батче
        loss = loss_fn(y_pred, y)                 # вычисляем ошибку проноза/значение функции потерь

        # Backpropagation
        loss.backward()                           # Вычисляем градиенты
        optimizer.step()                          # Обновляем веса по градиентам

        if print_loss:
            if batch % 10 == 0:
                loss, current = loss.item(), (batch + 1) * len(X)
                print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


In [ ]:
class ExtractTensor(nn.Module):
    def forward(self, x):
        # Если x - кортеж, берем первый элемент
        return x[0] if isinstance(x, tuple) else x


In [ ]:
# MLP с одним скрытым слоем
hidden_state = 30

model = nn.Sequential(
    nn.RNN(n_lags, hidden_state, num_layers=1, nonlinearity='tanh', bidirectional=False),
	# nn.GRU(n_lags, hidden_state, num_layers=1, bidirectional=False),
	# nn.LSTM(n_lags, hidden_state, num_layers=1, bidirectional=False)
	ExtractTensor(),
    nn.Linear(hidden_state, fh)
).to(device)


In [ ]:
# Число параметров модели
for params in model.parameters():
  print(params.size())

In [ ]:
# Обучение
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 10

for epoch in range(epochs):
	print(f"Epoch {epoch+1}\n-------------------------------")
	train(train_dataloader, model, loss_fn, optimizer)

print("Done!")

In [ ]:
inputs = torch.Tensor(series[-n_lags:]).unsqueeze(0) # переведём в тензор (1, n_lags)

model.eval()
with torch.no_grad():  # Отключаем вычисление градиентов
	outputs = model(inputs) # получаем тензор размера (1, fh)

outputs.squeeze() # переведём в одномерный тензор

In [ ]:
last_obs = 50

y_pred = outputs.squeeze().numpy()

plt.plot(np.arange(len(series)-last_obs, len(series)), series[-last_obs:])
plt.plot(np.arange(len(series), len(series)+fh), y_pred)
plt.show()